<a href="https://colab.research.google.com/github/LGBlack098/mi-repositorio-ed1/blob/main/Ejercicio_5_Desaf%C3%ADo_Repositorio_Estructuras_de_Datos_INF_220_UAGRM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Estructuras de Datos - INF 220 **[UAGRM]**
## Unidad I: Modelos de Representación de Datos
### Desafío: Capa de Repositorio (Backend Intercambiable)

**Objetivos:**
- Aplicar el principio de Inversión de Dependencias (DIP) y el patrón arquitectónico **Repository**.
- Crear una abstracción de persistencia (`PersistenciaBackend`) que permita cambiar el medio de almacenamiento subyacente (JSON, Pickle, BD, etc.) sin modificar la lógica del repositorio ni el código cliente.
- Manejar escenarios reales como la inicialización ante archivos inexistentes (`FileNotFoundError`).


### 1. Modelo de Dominio y Métodos Auxiliares


In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass, asdict
import json
import os
import pickle

@dataclass
class Estudiante:
    nombre: str
    nota: float
    grupo: str

def guardar_json(estudiantes, archivo):
    with open(archivo, "w", encoding="utf-8") as f:
        json.dump([asdict(e) for e in estudiantes], f, ensure_ascii=False, indent=2)

def cargar_json(archivo):
    with open(archivo, "r", encoding="utf-8") as f:
        datos = json.load(f)
    return [Estudiante(**d) for d in datos]

def guardar_pickle(estudiantes, archivo):
    with open(archivo, "wb") as f:
        pickle.dump(estudiantes, f)

def cargar_pickle(archivo):
    with open(archivo, "rb") as f:
        return pickle.load(f)


### 2. Interfaz Abstracta y Backends Concretos (`JSONBackend` y `PickleBackend`)


In [ ]:
class PersistenciaBackend(ABC):
    @abstractmethod
    def guardar(self, estudiantes, archivo):
        """Guarda la lista de estudiantes en el archivo indicado."""
        pass

    @abstractmethod
    def cargar(self, archivo):
        """Carga y devuelve la lista de estudiantes del archivo indicado."""
        pass


class JSONBackend(PersistenciaBackend):
    def guardar(self, estudiantes, archivo):
        guardar_json(estudiantes, archivo)

    def cargar(self, archivo):
        return cargar_json(archivo)


class PickleBackend(PersistenciaBackend):
    def guardar(self, estudiantes, archivo):
        guardar_pickle(estudiantes, archivo)

    def cargar(self, archivo):
        return cargar_pickle(archivo)


### 3. Implementación del Repositorio de Estudiantes


In [ ]:
class RepositorioEstudiantes:
    """Repositorio con backend de persistencia intercambiable (JSON/Pickle)."""

    def __init__(self, backend: PersistenciaBackend, archivo: str):
        self._backend = backend
        self._archivo = archivo
        self._estudiantes = []

    def agregar(self, estudiante: Estudiante):
        self._estudiantes.append(estudiante)

    def listar(self):
        return list(self._estudiantes)

    def guardar(self):
        self._backend.guardar(self._estudiantes, self._archivo)

    def cargar(self):
        # Si el archivo aún no existe (primera ejecución), se inicia con lista vacía
        try:
            self._estudiantes = self._backend.cargar(self._archivo)
        except FileNotFoundError:
            self._estudiantes = []


### 4. Demostración y Flujo de Ejecución


In [ ]:
def ejecutar_flujo(backend: PersistenciaBackend, archivo: str):
    repo = RepositorioEstudiantes(backend, archivo)
    repo.cargar()
    nuevo_est = Estudiante(f"Estudiante_{len(repo.listar()) + 1}", round(4.0 + len(repo.listar()) * 0.5, 1), "A")
    repo.agregar(nuevo_est)
    repo.guardar()

    print(f"Estado actual en '{archivo}' ({len(repo.listar())} registros):")
    for est in repo.listar():
        print("  -", est)
    print()

# Limpieza de archivos previos para prueba limpia
for f in ("repo_estudiantes.json", "repo_estudiantes.pkl"):
    if os.path.exists(f):
        os.remove(f)

print("=" * 60)
print("1. Primera ejecución con JSONBackend (Crea el archivo)")
print("=" * 60)
ejecutar_flujo(JSONBackend(), "repo_estudiantes.json")

print("=" * 60)
print("2. Segunda ejecución con JSONBackend (Carga y acumula)")
print("=" * 60)
ejecutar_flujo(JSONBackend(), "repo_estudiantes.json")

print("=" * 60)
print("3. Primera ejecución con PickleBackend (Crea el archivo binario)")
print("=" * 60)
ejecutar_flujo(PickleBackend(), "repo_estudiantes.pkl")

# Limpieza final
for f in ("repo_estudiantes.json", "repo_estudiantes.pkl"):
    if os.path.exists(f):
        os.remove(f)


### Conclusiones:
- **Desacoplamiento arquitectónico:** La capa de lógica de negocio o repositorio no tiene dependencias duras sobre formatos de archivo específicos ni librerías de terceros.
- **Extensibilidad:** Agregar un nuevo backend (por ejemplo, `SQLiteBackend` o `PostgreSQLBackend`) solo requiere implementar la interfaz `PersistenciaBackend` sin alterar ni una sola línea de `RepositorioEstudiantes`.
